# 3.6 — Feature Importance: Mutual Information

**Goal:** কোন categorical feature থেকে `churn` সম্পর্কে কতটা তথ্য পাওয়া যায়, তা একটি score দিয়ে মাপা এবং সব feature-কে বেশি থেকে কম গুরুত্বপূর্ণ ক্রমে সাজানো।

**Prerequisite:** আগের notebook-এর `df_full_train` এবং `categorical` list তৈরি করা থাকতে হবে।

## 1. Mutual Information (MI) কী?

ধরো, customer-এর `contract` type জানো। এই তথ্য থেকে customer churn করবে কি না, সে সম্পর্কে কিছু ধারণা পাওয়া যায়। কিন্তু শুধু `gender` জানলে churn সম্পর্কে খুব কম নতুন তথ্য পাওয়া যেতে পারে।

**Mutual Information (MI)** মাপে—কোনো feature জানার ফলে target `churn` সম্পর্কে কতটা তথ্য পাওয়া যায়, অর্থাৎ churn নিয়ে uncertainty কতটা কমে।

Information Theory-এর diagram-এ `X` ও `Y`-এর overlap-কে mutual information হিসেবে দেখানো হয়েছে: দুই variable-এর মধ্যে যে তথ্য shared, সেটাই MI। এখানে `X` হতে পারে একটি feature এবং `Y` হতে পারে `churn`।

- **MI বেশি** → feature জানলে churn সম্পর্কে তুলনামূলক বেশি তথ্য পাওয়া যায়।
- **MI প্রায় 0** → feature জানলে churn সম্পর্কে প্রায় কোনো নতুন তথ্য পাওয়া যায় না।
- **MI = 0** → এই dataset-এ feature ও target-এর মধ্যে mutual information নেই।

এটি association/information মাপে; feature যে সরাসরি churn ঘটায়, তা প্রমাণ করে না।

## 2. Risk Ratio থেকে পার্থক্য

আগের অংশে `risk ratio` দিয়ে একটি feature-এর category-গুলোর churn rate global churn rate-এর সঙ্গে তুলনা করেছি। Mutual Information আলাদা: এটি **প্রতিটি categorical feature-এর জন্য একটি score** দেয়, তাই feature-গুলোকে একই scale-এ rank করা যায়।

উদাহরণ হিসেবে, এই dataset-এ `contract`-এর MI প্রায় `0.0983`, কিন্তু `gender`-এর MI প্রায় `0.0001`। তাই `contract` churn সম্পর্কে অনেক বেশি তথ্য দেয়; `gender` খুব কম দেয়।

## 3. Scikit-learn function import

In [ ]:
from sklearn.metrics import mutual_info_score

`mutual_info_score(x, y)` দুইটি categorical variable-এর mutual information score বের করে। আমাদের ক্ষেত্রে একটি variable হলো feature এবং অন্যটি হলো `df_full_train.churn`। Argument-এর order বদলালেও score একই থাকে, কারণ MI symmetric।

## 4. সব categorical feature-এর MI বের করা

একেকটি column-এর জন্য আলাদা করে code না লিখে, একটি function তৈরি করে `apply()` দিয়ে সব categorical column-এ চালাব।

In [ ]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

**Function-এর কাজ:**

- `series` হলো একবারে একটি categorical column, যেমন `contract` বা `gender`।
- `mutual_info_score(series, df_full_train.churn)` ওই column-এর সঙ্গে `churn`-এর MI score বের করে।
- function শুধু score return করে, যাতে `apply()` প্রতিটি column-এর জন্য একই calculation চালাতে পারে।

In [ ]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

### Code-টি কীভাবে কাজ করে?

1. `df_full_train[categorical]` → শুধু categorical feature-গুলোর column নেয়; target `churn`-কে feature list-এ রাখা হয়নি।
2. `.apply(mutual_info_churn_score)` → প্রতিটি column-এর ওপর function চালিয়ে একটি MI score বের করে।
3. `mi` → feature name-কে index এবং MI score-কে value হিসেবে রাখে।
4. `.sort_values(ascending=False)` → score বড় থেকে ছোট সাজায়, তাই উপরের feature-গুলো churn সম্পর্কে বেশি তথ্য দেয়।

## 5. এই dataset-এর result কীভাবে পড়বে?

| Feature | MI score (প্রায়) | Interpretation |
|---|---:|---|
| `contract` | 0.098320 | সবচেয়ে বেশি তথ্য দেয় |
| `onlinesecurity` | 0.063085 | গুরুত্বপূর্ণ signal |
| `techsupport` | 0.061032 | গুরুত্বপূর্ণ signal |
| `internetservice` | 0.055868 | গুরুত্বপূর্ণ signal |
| `paymentmethod` | 0.043210 | কিছু তথ্য দেয় |
| `partner` | 0.009968 | তুলনামূলক কম তথ্য দেয় |
| `phoneservice` | 0.000229 | প্রায় কোনো তথ্য দেয় না |
| `gender` | 0.000117 | প্রায় কোনো তথ্য দেয় না |

Score-এর মান dataset ও data distribution অনুযায়ী নির্ভর করে। এই output-এ মূলত feature-গুলোর **আপেক্ষিক ranking** দেখব; score-কে percentage হিসেবে পড়বে না।

## Quick Recap

- **Mutual Information:** feature জানা থাকলে target সম্পর্কে কতটা তথ্য পাওয়া যায়।
- **High MI:** churn বোঝার জন্য feature-টি বেশি informative।
- **Low MI / 0-এর কাছাকাছি:** feature-টি churn সম্পর্কে কম তথ্য দেয়।
- `mutual_info_score(series, df_full_train.churn)` → একটি feature-এর MI score।
- `apply(mutual_info_churn_score)` → সব categorical feature-এর জন্য একই কাজ করে।
- `sort_values(ascending=False)` → সবচেয়ে informative feature-কে উপরে আনে।
- পরের অংশে numerical feature-এর জন্য **correlation** দেখা হবে।